# 05 — Backbone Screen on Fold 0

Mục tiêu: screen nhanh các backbone pretrained trên **cùng fold 0** trước khi tốn thời gian chạy 5-fold.

- Internal holdout 770 ảnh vẫn **không được dùng**.
- So sánh chính bằng official competition score `P` tại threshold 0.50.
- Threshold tuning chỉ để tham khảo ở bước screen.
- Candidates: **ConvNeXt-Tiny** và **EfficientNet-B2**.
- DenseNet121 fold 0 trước đó: `P@0.5 ≈ 0.98515` dùng làm mốc tham chiếu.


In [ ]:
import os
os.environ['TORCH_HOME'] = r'D:\AI_Cache\torch'

from pathlib import Path
import json, time, random, gc
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score

SEED=2026
PROJECT_ROOT=Path(r'D:\medicine')
MANIFEST_PATH=PROJECT_ROOT/'split_outputs'/'split_manifest.csv'
OUTPUT_DIR=PROJECT_ROOT/'screen_outputs'/'fold0_backbones'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

FOLD=0; IMAGE_SIZE=224; EPOCHS=7; PATIENCE=2
LR=1e-4; WEIGHT_DECAY=1e-4; NUM_WORKERS=0
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP=DEVICE.type=='cuda'
torch.backends.cudnn.benchmark = USE_AMP

print('PyTorch:',torch.__version__)
print('Device:',DEVICE)
if USE_AMP: print('GPU:',torch.cuda.get_device_name(0))
print('TORCH_HOME:',os.environ['TORCH_HOME'])
print('Manifest exists:',MANIFEST_PATH.exists())


In [ ]:
manifest=pd.read_csv(MANIFEST_PATH)
dev=manifest[manifest['split']=='dev'].copy()
holdout=manifest[manifest['split']=='holdout'].copy()
train_df=dev[dev['cv_fold']!=FOLD].copy().reset_index(drop=True)
val_df=dev[dev['cv_fold']==FOLD].copy().reset_index(drop=True)
assert len(train_df)==3489 and len(val_df)==873 and len(holdout)==770
assert set(train_df.sha256).isdisjoint(set(val_df.sha256))
assert set(dev.sha256).isdisjoint(set(holdout.sha256))
print('Train:',len(train_df),train_df.label.value_counts().to_dict())
print('Val:',len(val_df),val_df.label.value_counts().to_dict())
print('SEALED holdout:',len(holdout))


In [ ]:
class PadToSquare:
    def __call__(self,img):
        w,h=img.size; s=max(w,h)
        l=(s-w)//2; r=s-w-l; t=(s-h)//2; b=s-h-t
        return ImageOps.expand(img,border=(l,t,r,b),fill=0)

MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
train_tf=transforms.Compose([
    PadToSquare(),transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.RandomAffine(degrees=5,translate=(0.02,0.02),scale=(0.95,1.05),fill=0),
    transforms.ColorJitter(brightness=0.08,contrast=0.08),
    transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
val_tf=transforms.Compose([
    PadToSquare(),transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.ToTensor(),transforms.Normalize(MEAN,STD)])

L2I={'Normal':0,'Pneumonia':1}
class XrayDataset(Dataset):
    def __init__(self,frame,tf): self.frame=frame.reset_index(drop=True); self.tf=tf
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i]
        with Image.open(row.path) as im: im=im.convert('RGB')
        return self.tf(im),L2I[row.label],row.filename

train_ds=XrayDataset(train_df,train_tf); val_ds=XrayDataset(val_df,val_tf)


In [ ]:
def metric(y,prob,t=0.5):
    y=np.asarray(y,int); prob=np.asarray(prob,float); pred=(prob>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    acc=(tp+tn)/len(y); se=tp/max(tp+fn,1); sp=tn/max(tn+fp,1)
    f1=f1_score(y,pred,pos_label=1,zero_division=0)
    P=.30*acc+.25*se+.20*sp+.25*f1
    auc=roc_auc_score(y,prob)
    return {'P':float(P),'Accuracy':float(acc),'Sensitivity':float(se),'Specificity':float(sp),'F1':float(f1),'AUROC':float(auc),'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),'threshold':float(t)}

@torch.no_grad()
def predict(model,loader):
    model.eval(); ys=[]; ps=[]; names=[]
    for x,y,n in loader:
        x=x.to(DEVICE,non_blocking=True)
        with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
            logits=model(x)
        p=torch.softmax(logits.float(),1)[:,1]
        ys.extend(y.numpy()); ps.extend(p.cpu().numpy()); names.extend(list(n))
    return np.asarray(ys),np.asarray(ps),names

def build_model(name):
    if name=='convnext_tiny':
        m=models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        m.classifier[2]=nn.Linear(m.classifier[2].in_features,2)
        batch_size=16
    elif name=='efficientnet_b2':
        m=models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
        m.classifier[1]=nn.Linear(m.classifier[1].in_features,2)
        batch_size=16
    else: raise ValueError(name)
    return m.to(DEVICE),batch_size


## Train candidates

Nếu model đầu tiên gặp `CUDA out of memory`, giảm `batch_size=16` trong `build_model()` xuống `8` cho model đó rồi chạy lại cell này.

In [ ]:
CANDIDATES=['convnext_tiny','efficientnet_b2']
all_results=[]

for model_name in CANDIDATES:
    print('\n'+'='*75); print('MODEL:',model_name); print('='*75)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if USE_AMP: torch.cuda.manual_seed_all(SEED)
    model,batch_size=build_model(model_name)
    train_loader=DataLoader(train_ds,batch_size=batch_size,shuffle=True,num_workers=NUM_WORKERS,pin_memory=USE_AMP)
    val_loader=DataLoader(val_ds,batch_size=batch_size,shuffle=False,num_workers=NUM_WORKERS,pin_memory=USE_AMP)

    c=train_df.label.value_counts(); n0=int(c.Normal); n1=int(c.Pneumonia); total=n0+n1
    cw=torch.tensor([total/(2*n0),total/(2*n1)],dtype=torch.float32,device=DEVICE)
    criterion=nn.CrossEntropyLoss(weight=cw)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)
    scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)

    best_score=-1; best_state=None; best_epoch=0; stale=0; hist=[]
    for epoch in range(1,EPOCHS+1):
        start=time.time(); model.train(); loss_sum=0; seen=0
        for x,y,_ in train_loader:
            x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
                logits=model(x); loss=criterion(logits,y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            loss_sum+=float(loss.item())*len(x); seen+=len(x)
        scheduler.step()
        yv,pv,_=predict(model,val_loader); m=metric(yv,pv,0.5)
        hist.append({'epoch':epoch,'loss':loss_sum/seen,**m})
        print(f"Epoch {epoch:02d} | loss={loss_sum/seen:.4f} | P={m['P']:.5f} | A={m['Accuracy']:.5f} | Se={m['Sensitivity']:.5f} | Sp={m['Specificity']:.5f} | F1={m['F1']:.5f} | {time.time()-start:.0f}s")
        if m['P']>best_score+1e-6:
            best_score=m['P']; best_epoch=epoch; stale=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            stale+=1
            if stale>=PATIENCE: print('Early stopping'); break

    model.load_state_dict(best_state)
    yv,pv,names=predict(model,val_loader)
    fixed=metric(yv,pv,0.5)
    scan=pd.DataFrame([metric(yv,pv,t) for t in np.round(np.arange(0.10,0.901,0.005),3)])
    row=scan.sort_values(['P','threshold'],ascending=[False,True]).iloc[0]
    tuned=metric(yv,pv,float(row.threshold))
    result={'model':model_name,'best_epoch':best_epoch,'batch_size':batch_size,'metrics_at_0_5':fixed,'single_fold_best_threshold':float(row.threshold),'single_fold_tuned_metrics':tuned}
    all_results.append(result)

    pd.DataFrame(hist).to_csv(OUTPUT_DIR/f'{model_name}_history.csv',index=False)
    pd.DataFrame({'filename':names,'y_true':yv,'p_pneumonia':pv}).to_csv(OUTPUT_DIR/f'{model_name}_predictions.csv',index=False)
    scan.to_csv(OUTPUT_DIR/f'{model_name}_threshold_scan.csv',index=False)
    torch.save({'model':model_name,'best_epoch':best_epoch,'state_dict':best_state},OUTPUT_DIR/f'{model_name}_best.pt')
    print(json.dumps(result,indent=2))

    del model,optimizer,scheduler,scaler,best_state,train_loader,val_loader
    gc.collect()
    if USE_AMP: torch.cuda.empty_cache()


## Ranking

In [ ]:
rows=[]
for r in all_results:
    m=r['metrics_at_0_5']; tm=r['single_fold_tuned_metrics']
    rows.append({'model':r['model'],'best_epoch':r['best_epoch'],'P_at_0_5':m['P'],'AUROC':m['AUROC'],'Sensitivity':m['Sensitivity'],'Specificity':m['Specificity'],'F1':m['F1'],'best_threshold':r['single_fold_best_threshold'],'P_tuned_fold0':tm['P']})
ranking=pd.DataFrame(rows)
dense_ref=pd.DataFrame([{'model':'densenet121_reference','best_epoch':6,'P_at_0_5':0.9851505114783624,'AUROC':0.9988618382699942,'Sensitivity':0.9800613496932515,'Specificity':0.9909502262443439,'F1':0.988399071925754,'best_threshold':0.205,'P_tuned_fold0':0.9872839166987125}])
ranking=pd.concat([dense_ref,ranking],ignore_index=True).sort_values('P_at_0_5',ascending=False)
display(ranking)
ranking.to_csv(OUTPUT_DIR/'backbone_screen_ranking.csv',index=False)
with open(OUTPUT_DIR/'backbone_screen_results.json','w',encoding='utf-8') as f: json.dump(all_results,f,indent=2)
print('Saved to:',OUTPUT_DIR)


## Decision rule

- Backbone nào thắng hoặc bám sát DenseNet121 fold 0 và có lỗi bổ sung khác DenseNet sẽ được cân nhắc chạy 5-fold.
- Không chọn chỉ theo threshold tuned trên fold 0.
- Sau screening, ưu tiên 1–2 model tốt nhất để tiết kiệm GPU-hours cho các ablation có giá trị hơn.
